# Testing GWAI on GPT-2 (Greater-Than Task)

Compares GWAI (k=0 and k=1) against information-flow-routes and EAP-IG on the greater-than circuit.

In [1]:
from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizer
from transformer_lens import HookedTransformer

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline
from eap.attribute import attribute

## Dataset and Metrics

In [2]:
def collate_EAP(xs):
    clean, corrupted, labels = zip(*xs)
    clean = list(clean)
    corrupted = list(corrupted)
    return clean, corrupted, labels

class EAPDataset(Dataset):
    def __init__(self, filepath):
        self.df = pd.read_csv(filepath)

    def __len__(self):
        return len(self.df)

    def shuffle(self):
        self.df = self.df.sample(frac=1)

    def head(self, n: int):
        self.df = self.df.head(n)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        return row['clean'], row['corrupted'], row['label']

    def to_dataloader(self, batch_size: int):
        return DataLoader(self, batch_size=batch_size, collate_fn=collate_EAP)

def get_logit_positions(logits: torch.Tensor, input_length: torch.Tensor):
    batch_size = logits.size(0)
    idx = torch.arange(batch_size, device=logits.device)
    logits = logits[idx, input_length - 1]
    return logits

def get_prob_diff(tokenizer: PreTrainedTokenizer):
    year_indices = torch.tensor([tokenizer(f'{year:02d}').input_ids[0] for year in range(100)])

    def prob_diff(logits, clean_logits, input_length, labels, mean=True, loss=False):
        logits = get_logit_positions(logits, input_length)
        probs = torch.softmax(logits, dim=-1)[:, year_indices]
        results = []
        for prob, year in zip(probs, labels):
            results.append(prob[year + 1:].sum() - prob[:year + 1].sum())
        results = torch.stack(results)
        if loss:
            results = -results
        if mean:
            results = results.mean()
        return results
    return prob_diff

## Load Model and Data

In [4]:
model_name = 'gpt2-small'
device = 'cpu'

model = HookedTransformer.from_pretrained(model_name, device=device)
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True

metric = get_prob_diff(model.tokenizer)

dataset = EAPDataset('greater_than_data.csv')
dataloader = dataset.to_dataloader(batch_size=120)

Loaded pretrained model gpt2-small into HookedTransformer


## Baseline: Full Model Performance

In [ ]:
baseline = evaluate_baseline(model, dataloader, metric, quiet=False)
baseline_mean = baseline.mean().item()
print(f"Full model prob_diff: {baseline_mean:.4f}")

## Attribution: GWAI k=0 (Pure ALTI), GWAI k=1, EAP-IG, Information Flow Routes

In [ ]:
results = {}

# GWAI k=0 (should match information-flow-routes)
print("=== GWAI k=0 ===")
graph_gwai0 = Graph.from_model(model)
attribute(model, graph_gwai0, dataloader, metric, method='GWAI', gwai_k=0)
graph_gwai0.apply_topn(200, absolute=True)
score_gwai0 = evaluate_graph(model, graph_gwai0, dataloader, metric, quiet=False).mean().item()
results['GWAI k=0'] = score_gwai0
print(f"GWAI k=0 prob_diff: {score_gwai0:.4f}\n")

# GWAI k=1
print("=== GWAI k=1 ===")
graph_gwai1 = Graph.from_model(model)
attribute(model, graph_gwai1, dataloader, metric, method='GWAI', gwai_k=1)
graph_gwai1.apply_topn(200, absolute=True)
score_gwai1 = evaluate_graph(model, graph_gwai1, dataloader, metric, quiet=False).mean().item()
results['GWAI k=1'] = score_gwai1
print(f"GWAI k=1 prob_diff: {score_gwai1:.4f}\n")

# Information Flow Routes
print("=== Information Flow Routes ===")
graph_ifr = Graph.from_model(model)
attribute(model, graph_ifr, dataloader, metric, method='information-flow-routes')
graph_ifr.apply_topn(200, absolute=True)
score_ifr = evaluate_graph(model, graph_ifr, dataloader, metric, quiet=False).mean().item()
results['Info Flow Routes'] = score_ifr
print(f"Info Flow Routes prob_diff: {score_ifr:.4f}\n")

# EAP-IG
print("=== EAP-IG ===")
graph_eap = Graph.from_model(model)
attribute(model, graph_eap, dataloader, metric, method='EAP-IG-inputs', ig_steps=5)
graph_eap.apply_topn(200, absolute=True)
score_eap = evaluate_graph(model, graph_eap, dataloader, metric, quiet=False).mean().item()
results['EAP-IG'] = score_eap
print(f"EAP-IG prob_diff: {score_eap:.4f}")

## Summary

In [ ]:
print(f"{'Method':<20} {'prob_diff':>10} {'% of baseline':>15}")
print("-" * 47)
for name, score in results.items():
    pct = score / baseline_mean * 100 if baseline_mean != 0 else float('nan')
    print(f"{name:<20} {score:>10.4f} {pct:>14.1f}%")
print("-" * 47)
print(f"{'Full model':<20} {baseline_mean:>10.4f} {'100.0%':>15}")